# Raport - atak Bit Flip na algorytm AES w trybie CBC

Filip Horst 311257 - Laboratorium Bezpieczeństwo Aplikacji Internetowych i Mobilnych

Zadanie domowe - wykonanie ataku bitflip na przykładowym pliku aes_bitflip.py dostarczonym w materiałach

# Wstęp
Raport zawiera kilka części: 
* ręczne wykonanie pojedynczej podmiany w celu udowodnienia zrozumienia ataku
* automatyzację i sprawdzenie ataku skryptami
* prezentację skutków ubocznych (uszkodzonego bloku)


W kodzie posługuję się skrótami:
* enc - kryptogram
* msg - tekst jawny
* int - stan wewnętrzny dekodera

Każdy z nich może mieć sufiks:
* b - do oznaczenia, że zmienna w formie bajtów
* #i - numer bloku (i)
* pos - oznacza, że zmienna zawiera tylko jeden znak z danej pozycji, a nie cały blok

In [ ]:
#Funkcje pomocnicze do ładnego wyświetlania
def xorprint(bt):
    return bt.to_bytes().hex()
def hexprint(hx):
    x = []
    for i in range(0,len(hx),2):
        x.append(hx[i]+hx[i+1])
    return ' '.join(x)

# Wykonanie - atak przeprowadzony ręcznie

In [225]:
full_enc = """f6472ef9ee746bb26d0e81126e0c2073
fcba60e430d4393fffedabb9c2063512
170310d3f9f6cf27203951198559da34
bf2f937bdfcda59663db7e668418db48
0533224ff12e6b98ea039996b2054b55
b6f89a9fae8653bac6aca5268c929a9f
d41f2f0fc42df40cdb25e9926d88ed15"""

full_msg = '{"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"bartosz.chaber@pw.edu.pl","role":"usr"}'
print(full_enc.replace("\n",""))

f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da34bf2f937bdfcda59663db7e668418db480533224ff12e6b98ea039996b2054b55b6f89a9fae8653bac6aca5268c929a9fd41f2f0fc42df40cdb25e9926d88ed15


Wykonanie programu z podstawowym tokenem skutkuje odszyfrowaniem, ale brakiem dostępu:

```python3 .\aes_bitflip.py
b'Plaintext token: {"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"bartosz.chaber@pw.edu.pl","role":"usr"}'
b'Encrypted token: 'f6472ef9ee746bb26d0e81126e0c2073
fcba60e430d4393fffedabb9c2063512
170310d3f9f6cf27203951198559da34
bf2f937bdfcda59663db7e668418db48
0533224ff12e6b98ea039996b2054b55
b6f89a9fae8653bac6aca5268c929a9f
d41f2f0fc42df40cdb25e9926d88ed15
Your token in hex: f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da34bf2f937bdfcda59663db7e668418db480533224ff12e6b98ea039996b2054b55b6f89a9fae8653bac6aca5268c929a9fd41f2f0fc42df40cdb25e9926d88ed15
Only for admins!```

Atak bitflip zakłada znajomość kryptogramu oraz tekstu jawnego. Można je podzielić na bloki w celu lepszej wizualizacji, które dane sobie odpowiadają.

In [226]:
for i in range(6):
    print(f'Wiadomość msg#{i}: {full_msg[i*16:i*16+16]}  Kryptogram enc#{i}: {full_enc.split('\n')[i]} ')

Wiadomość msg#0: {"uid:"20a03945-  Kryptogram enc#0: f6472ef9ee746bb26d0e81126e0c2073 
Wiadomość msg#1: e544-4f45-a31e-2  Kryptogram enc#1: fcba60e430d4393fffedabb9c2063512 
Wiadomość msg#2: ddc2b5cc6fc","us  Kryptogram enc#2: 170310d3f9f6cf27203951198559da34 
Wiadomość msg#3: ername":"bartosz  Kryptogram enc#3: bf2f937bdfcda59663db7e668418db48 
Wiadomość msg#4: .chaber@pw.edu.p  Kryptogram enc#4: 0533224ff12e6b98ea039996b2054b55 
Wiadomość msg#5: l","role":"usr"}  Kryptogram enc#5: b6f89a9fae8653bac6aca5268c929a9f 


By wykonać atak polegający na zmianie roli należy zainteresować się blokiem msg#5. Przy jego deszyfracji biorą udział:
* blok enc#5 - wejście do szyfru blokowego, z którego wychodzi stan wewnętrzny int#5
* blok enc#4 - XOR-owany ze stanem wewnętrznym int#5 daje msg#5
* blok msg#5 - wynik deszyfracji, fragment tekstu jawnego, równy int#5 XOR enc#4

In [227]:
enc4 = '0533224ff12e6b98ea039996b2054b55' #Blok #4 kryptogramu enc
msg5 = 'l","role":"usr"}' #Blok #5 wiadomości jawnej msg

encb4 = bytes.fromhex(enc4) #Zamiana na bajty (sufiks b)
msgb5 = msg5.encode()   #Zamiana na bajty (sufiks b)

pos = 11 #Pozycja do podmiany znaku - fragment ustawień ataku
print(f'Blok kryptogramu: {hexprint(enc4)}\nBlok wiadomości: {msg5}\nAtakowany znak: {chr(msgb5[pos])}\nOdpowiadający fragment kryptogramu (HEX): {enc4[2*pos:2*pos+2]}')

target = 'a' #Nowy znak, który ma się pokazać po zdekodowaniu zmodyfikowanego kryptogramu


Blok kryptogramu: 05 33 22 4f f1 2e 6b 98 ea 03 99 96 b2 05 4b 55
Blok wiadomości: l","role":"usr"}
Atakowany znak: u
Odpowiadający fragment kryptogramu (HEX): 96


Odkrycie stanu wewnętrznego jest oparte o właściwości operacji XOR: int#5 = msg#5 XOR enc#4

In [228]:
intb5pos = msgb5[pos] ^ encb4[pos]
print(f"Fragment stanu wewnętrznego int (HEX): {xorprint(msgb5[pos])} XOR {xorprint(encb4[pos])} = {xorprint(intb5pos)}")

Fragment stanu wewnętrznego int (HEX): 75 XOR 96 = e3


Modyfikacja bloku poprzedzającego tak, aby XOR z odkrytym stanem wewnętrznym dał nowy znak docelowy 

In [229]:
new_encb4pos = intb5pos ^ ord(target)
print(f"Nowy fragment kryptogramu (HEX): {xorprint(intb5pos)} XOR {ord(target)} = {xorprint(new_encb4pos)}")

Nowy fragment kryptogramu (HEX): e3 XOR 97 = 82


Sprawdzenie poprzez wykonanie XOR nowego enc#4 ze stanem wewnętrznym int#5

In [230]:
targettest = (new_encb4pos ^ intb5pos)
print(f'{xorprint(new_encb4pos)} XOR {xorprint(intb5pos)} = {targettest} ({chr(targettest)})')

82 XOR e3 = 97 (a)


Co jest zgodne z celem target, czyli atak przebiegł poprawnie. Pozostaje zmodyfikować odpowiedni bajt kryptogramu enc#4

In [231]:
new_encb4 = bytes(bytearray(encb4[:pos]) + new_encb4pos.to_bytes() + encb4[pos+1:])
print(f"Przed: {hexprint(encb4.hex())}\nPo:    {hexprint(new_encb4.hex())}")

Przed: 05 33 22 4f f1 2e 6b 98 ea 03 99 96 b2 05 4b 55
Po:    05 33 22 4f f1 2e 6b 98 ea 03 99 82 b2 05 4b 55


# Wykonanie - atak z użyciem skryptu

Opisany w poprzedniej sekcji proces można spakować w funkcję, aby ułatwić wykonanie dla wielu znaków

In [232]:

def bitflip_attack(enc, msg, pos, target):
    """
    Wykonuje atak bitflip na i-ty fragment tekstu jawnego
    :param str enc: Blok kryptogramu poprzedzającego fragment tekstu który chcemy zmienić (blok enc#i-1 dla msg#i). W formacie HEX typu str
    :param str msg: Fragment jawny, który chcemy zmodyfikować msg#i
    :param int pos: Pozycja w bloku zmienianego znaku
    :param str target: Nowy znak 
    
    Zwraca nowy blok enc#i-1
    """
    msgb = msg.encode() #str -> bytes
    encb = bytes.fromhex(enc) #Hex str -> bytes
    print(f'Bitflip pos {pos}: {chr(msgb[pos])} -> {target}...')
    intbpos = msgb[pos] ^ encb[pos] #tekst jawny XOR kryptogram = stan wewnetrzny
    new_encbpos = intbpos ^ ord(target)  #stan wewnetrzny XOR nowy znak = nowy fragment kryptogramu
    new_enc = bytes(bytearray(encb[:pos]) + new_encbpos.to_bytes() + encb[pos+1:]).hex() #zlozenie nowego kryptogramu
 
    return new_enc
    
    

In [233]:

enc = '0533224ff12e6b98ea039996b2054b55' #enc#4
msg = full_msg[5*16:5*16+16] #msg#5
print("START")
print(hexprint(enc))


print('_______1________')
enc = bitflip_attack(enc, msg, 11, 'a')
print(hexprint(enc))

print('_______2________')
enc = bitflip_attack(enc, msg, 12, 'd')
print(hexprint(enc))

print('_______3________')
enc = bitflip_attack(enc, msg, 13, 'm')
print(hexprint(enc))

START
05 33 22 4f f1 2e 6b 98 ea 03 99 96 b2 05 4b 55
_______1________
Bitflip pos 11: u -> a...
05 33 22 4f f1 2e 6b 98 ea 03 99 82 b2 05 4b 55
_______2________
Bitflip pos 12: s -> d...
05 33 22 4f f1 2e 6b 98 ea 03 99 82 a5 05 4b 55
_______3________
Bitflip pos 13: r -> m...
05 33 22 4f f1 2e 6b 98 ea 03 99 82 a5 1a 4b 55


Ostatnim etapem jest podmiana starego bloku na zmodyfikowany i sprawdzenie odpowiedzi programu

In [234]:
x = full_enc.split('\n')
x[4] = enc
bitflip_token = ''.join(x)
print(bitflip_token)

f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da34bf2f937bdfcda59663db7e668418db480533224ff12e6b98ea039982a51a4b55b6f89a9fae8653bac6aca5268c929a9fd41f2f0fc42df40cdb25e9926d88ed15


```
PS B:\prg\L25\baim\BAIM_JuiceShop_25L\baim_pentests\baim_assets> python3 .\aes_bitflip.py
b'Plaintext token: {"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"bartosz.chaber@pw.edu.pl","role":"usr"}'
b'Encrypted token: 'f6472ef9ee746bb26d0e81126e0c2073
fcba60e430d4393fffedabb9c2063512
170310d3f9f6cf27203951198559da34
bf2f937bdfcda59663db7e668418db48
0533224ff12e6b98ea039996b2054b55
b6f89a9fae8653bac6aca5268c929a9f
d41f2f0fc42df40cdb25e9926d88ed15
Your token in hex: f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da34bf2f937bdfcda59663db7e668418db480533224ff12e6b98ea039982a51a4b55b6f89a9fae8653bac6aca5268c929a9fd41f2f0fc42df40cdb25e9926d88ed15
You are an admin!
```

Modyfikacje sprawiły, że token został odszyfrowany i przyznane zostały uprawnienia admina: atak się powiódł

# Analiza efektów ubocznych

Funkcje pomocnicze pochodzące z aes_bitflip.py. Oczywiście nie były używane w ataku.

In [235]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from binascii import b2a_hex, a2b_hex

key = a2b_hex("d41860300bb37d0f252cdfe84e658fce275eac6c56e142f74014403657ea4da1")
iv  = a2b_hex("fffeafc74c4cdcbfe73e6222bde88a65")


def decrypt_data(enc):
    cipher = AES.new(key, AES.MODE_CBC, iv)
    msg = unpad(cipher.decrypt(enc), cipher.block_size, style='pkcs7')
    return msg

Scenariusz bez modyfikacji

In [236]:
data = a2b_hex(full_enc.replace('\n',''))
decrypt_data(data)

b'{"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"bartosz.chaber@pw.edu.pl","role":"usr"}'

Scenariusz po modyfikacjach bitflip

In [237]:
data = a2b_hex(bitflip_token)
decrypt_data(data)

b'{"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"bartosz%\xec]j\x87\x07\xc2\x00!\xee\x9an\x94\xea\x99Al","role":"adm"}'

Porównanie obu tokenów po deszyfracji pokazuje skutki bitflip. W drugim przypadku (po ataku) zmianie uległa rola, co wyjaśnia przyznanie dostępu administratora w programie. Widoczny jest też skutek uboczny, czyli zamiana kawałka poprzedniego bloku na losowo wyglądające bajty.

Takie zjawisko jest spowodowane tym, że modyfikowany blok kryptogramu enc#i bierze udział w deszyfrowaniu zarówno bloku msg#i+1, jak i msg#i. W badanym przypadku zmieniany enc#4 służył do odkodowania msg#5, które było przez nas kontrolowane. Wprowadzane w tym procesie zmiany są również widoczne w msg#4, ale ponieważ w trakcie ataku nikogo one nie interesowały to ostatecznie wyglądają jak losowe uszkodzenia.

# Kodowanie innych danych
Przykład na innych danych potwierdzający sprawność skryptu

In [238]:
full_enc = """f6472ef9ee746bb26d0e81126e0c2073
fcba60e430d4393fffedabb9c2063512
170310d3f9f6cf27203951198559da34
43b9343385b7b6ba09dd6c7894a54ff1
d2f35113c594a9e4f97fb5917031a99f
2b3455c04809c45f25b71880632383b3"""

full_msg = '{"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"filip.horst@pw.edu.pl","role":"usr"}'

print(full_enc.replace("\n",""))

f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da3443b9343385b7b6ba09dd6c7894a54ff1d2f35113c594a9e4f97fb5917031a99f2b3455c04809c45f25b71880632383b3


In [239]:
enc = 'd2f35113c594a9e4f97fb5917031a99f' #enc#4
msg = full_msg[5*16:5*16+16] #msg#5
print("START")
print(hexprint(enc))


print('_______1________')
enc = bitflip_attack(enc, msg, 8, 'a')
print(hexprint(enc))

print('_______2________')
enc = bitflip_attack(enc, msg, 9, 'd')
print(hexprint(enc))

print('_______3________')
enc = bitflip_attack(enc, msg, 10, 'm')
print(hexprint(enc))

x = full_enc.split('\n')
x[4] = enc
bitflip_token = ''.join(x)
print(bitflip_token)

START
d2 f3 51 13 c5 94 a9 e4 f9 7f b5 91 70 31 a9 9f
_______1________
Bitflip pos 8: u -> a...
d2 f3 51 13 c5 94 a9 e4 ed 7f b5 91 70 31 a9 9f
_______2________
Bitflip pos 9: s -> d...
d2 f3 51 13 c5 94 a9 e4 ed 68 b5 91 70 31 a9 9f
_______3________
Bitflip pos 10: r -> m...
d2 f3 51 13 c5 94 a9 e4 ed 68 aa 91 70 31 a9 9f
f6472ef9ee746bb26d0e81126e0c2073fcba60e430d4393fffedabb9c2063512170310d3f9f6cf27203951198559da3443b9343385b7b6ba09dd6c7894a54ff1d2f35113c594a9e4ed68aa917031a99f2b3455c04809c45f25b71880632383b3


In [240]:
data = a2b_hex(bitflip_token)
decrypt_data(data)

b'{"uid:"20a03945-e544-4f45-a31e-2ddc2b5cc6fc","username":"filip.h\x8d\xa6c/\x17\x10\x1a\x14\x90S\xb9\xbe\xb7U\x8f\xd6"role":"adm"}'

Użycie takiego bitflip token pozwala na uzyskanie dostępu. Biorąc pod uwagę metodę przyznawania dostępu admina można użyć tokena dla dowolnych danych username - ważne, żeby było role:adm.

# Podsumowanie

Zawsze byłem przekonany (a przynajmniej do czasu odbycia kursu Kryptografia Stosowana), że tryb CBC zabezpiecza AES przed atakami, jednak jak się okazuje jest tak tylko do czasu gdy nie wystąpi żaden dodatkowy bit informacji, jaki tutaj stanowi zwracany kryptogram. Podobnie w padding oracle attack źródłem informacji są błędy lub czas wykonania. Takie pomniejsze zadanie jest wystarczające, żeby pokazać niebezpieczeństwo płynące z nieodpowiednich ustawień AES i przy kolejnym projekcie na pewno zdecyduję się na użycie trybów z autoryzacją CCM, czy GCM.